# DiffShield job runner
Set `JOB_IDS` in the first code cell, then **Run All** (or *Save Version -> Save & Run All*).
Nothing else needs editing. Jobs are defined in `kaggle/jobs.yaml` inside the code bundle.

Checklist before running: Internet **ON**; accelerator as the job needs (J00: none/CPU is enough; J01: **GPU T4 x2**);
Add Data: the code dataset (`diffshield-code`) and, for J01, the CelebAMask-HQ dataset (`ipythonx/celebamaskhq`).
No tokens or logins are needed or should be entered.

In [ ]:
# ---- the only cell you edit -------------------------------------------------------------
JOB_IDS      = ["J00"]      # run in this order, e.g. ["J01"] or ["J11", "J20a"]
JOB_SHARD    = None         # e.g. "0/2" to run only half of a job's items (other half on another account)
RESUME_FROM  = None         # path of an earlier session's output added as an input, e.g. "/kaggle/input/diffshield-j01-out"
PARAMS       = {}           # overrides of job params, e.g. {"n_images": 5} for a smoke test
FORCE        = False        # True: rerun jobs that already finished in this session
# -------------------------------------------------------------------------------------------

In [ ]:
import os, sys, json, shutil, zipfile, subprocess
from pathlib import Path

CODE_DIR = Path("/kaggle/working/DiffShield")

def find_bundle(root="/kaggle/input"):
    """Return a directory containing BUNDLE_INFO.json (Kaggle unzips uploaded zips) or a zip that contains it."""
    best = None
    for r, dirs, files in os.walk(root):
        depth = r[len(root):].count(os.sep)
        if "BUNDLE_INFO.json" in files:
            return ("dir", Path(r))
        for f in files:
            if f.startswith("diffshield-code") and f.endswith(".zip"):
                best = ("zip", Path(r) / f)
        if depth > 4:
            dirs[:] = []
    return best

if not (CODE_DIR / "scripts" / "run_job.py").exists():
    found = find_bundle()
    assert found, "code bundle not found under /kaggle/input: add the diffshield-code dataset to this notebook"
    kind, p = found
    if CODE_DIR.exists():
        shutil.rmtree(CODE_DIR)
    if kind == "dir":
        shutil.copytree(p, CODE_DIR)
    else:
        with zipfile.ZipFile(p) as z:
            z.extractall(CODE_DIR)
print("code at", CODE_DIR)
print(json.dumps({k: v for k, v in json.loads((CODE_DIR / "BUNDLE_INFO.json").read_text()).items() if k != "sha256"}, indent=2))

In [ ]:
cmd = [sys.executable, "scripts/run_job.py", "--jobs", ",".join(JOB_IDS), "--work", "/kaggle/working"]
if JOB_SHARD: cmd += ["--job_shard", JOB_SHARD]
if RESUME_FROM: cmd += ["--resume_from", RESUME_FROM]
if FORCE: cmd += ["--force"]
for k, v in PARAMS.items(): cmd += ["--set", f"{k}={json.dumps(v)}"]
print(" ".join(cmd))
env = dict(os.environ, PYTHONUNBUFFERED="1", PYTHONIOENCODING="utf-8")
p = subprocess.Popen(cmd, cwd=CODE_DIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout:
    print(line, end="")
p.wait()
print("run_job exit code:", p.returncode)

In [ ]:
# Summary of what was produced (download the *_outputs.zip files from the Output tab afterwards)
for f in sorted(Path("/kaggle/working").glob("*_outputs.zip")):
    print(f.name, round(f.stat().st_size / 1e6, 1), "MB")
sm = Path("/kaggle/working/job_runner_summary.json")
if sm.exists():
    for k, v in json.loads(sm.read_text()).items():
        print(k, "ok" if v.get("ok") else "FAILED", round(v["session_seconds"]), "s")
assert p.returncode == 0, "a job failed: see /kaggle/working/<JOB>/logs/ ; rerun the same cell to resume"